# nuScenes LiDAR Detection with MMDetection3D

This tutorial evaluates a PointPillars or CenterPoint LiDAR 3D object-detection model on the nuScenes `v1.0-mini` split using PerceptionMetrics. It covers loading the dataset, wrapping an MMDetection3D model, inspecting predictions in bird's-eye view, and evaluating the model.

It expects the nuScenes mini dataset in `examples/local/nuscenes/v1.0-mini` and the selected model config and checkpoint under `examples/local/model`.

In [ ]:
from copy import copy
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from mmdet3d.apis import init_model

from perceptionmetrics.datasets.nuScenes import NuScenesLiDARDetectionDataset
from perceptionmetrics.models.torch_detection import TorchLiDARDetectionModel

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "perceptionmetrics").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

LOCAL_DIR = PROJECT_ROOT / "examples" / "local"
DATASET_DIR = LOCAL_DIR / "nuscenes" / "v1.0-mini"
# Select one downloaded MMDetection3D model: "pointpillars" or "centerpoint".
MODEL_NAME = "centerpoint"

MODEL_FILES = {
    "pointpillars": (
        "nuscenes_pointpillars",
        "pointpillars_hv_fpn_sbn-all_8xb4-2x_nus-3d.py",
        "hv_pointpillars_fpn_sbn-all_4x8_2x_nus-3d_20210826_104936-fca299c1.pth",
    ),
    "centerpoint": (
        "nuscenes_centerpoint",
        "centerpoint_pillar02_second_secfpn_head-circlenms_8xb4-cyclic-20e_nus-3d.py",
        "centerpoint_02pillar_second_secfpn_circlenms_4x8_cyclic_20e_nus_20220811_031844-191a3822.pth",
    ),
}

if MODEL_NAME not in MODEL_FILES:
    raise ValueError(f"Unsupported MODEL_NAME: {MODEL_NAME}")

model_dir_name, config_name, checkpoint_name = MODEL_FILES[MODEL_NAME]
MODEL_DIR = LOCAL_DIR / "model" / model_dir_name
MMDET_CONFIG = MODEL_DIR / config_name
CHECKPOINT = MODEL_DIR / checkpoint_name


DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

if not DATASET_DIR.is_dir() or not MMDET_CONFIG.is_file() or not CHECKPOINT.is_file():
    raise FileNotFoundError("NuScenes dataset or selected model files not found.")

dataset = NuScenesLiDARDetectionDataset(
    dataset_dir=str(DATASET_DIR),
    version="v1.0-mini",
    split="val",
)
print(f"Validation samples: {len(dataset.dataset)}")

## PerceptionMetrics Model Metadata

The nuScenes adapter maps raw nuScenes categories to the ten detection classes used by the selected checkpoint. This cell writes the PerceptionMetrics model configuration, ontology, and label translation to `local/data/`.

The ontology uses the official nuScenes devkit RGB palette. `bus` and `pedestrian` combine several raw nuScenes categories, so the rigid-bus and adult-pedestrian colors are used as their representative colors.

In [ ]:
from nuscenes.utils.color_map import get_colormap

ARTIFACT_DIR = LOCAL_DIR / "data"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_CONFIG_PATH = ARTIFACT_DIR / f"nuscenes_{MODEL_NAME}_model_config.json"
MODEL_ONTOLOGY_PATH = ARTIFACT_DIR / f"nuscenes_{MODEL_NAME}_ontology.json"
TRANSLATION_PATH = ARTIFACT_DIR / f"nuscenes_to_{MODEL_NAME}_translation.json"
model_config = {
    "model_format": "mmdet3d",
    "confidence_threshold": 0.5,
    "iou_threshold": 0.5,
}

# Keep this order identical to the selected MMDetection3D model config.
MODEL_CLASS_NAMES = {
    "pointpillars": [
        "car", "truck", "trailer", "bus", "construction_vehicle",
        "bicycle", "motorcycle", "pedestrian", "traffic_cone", "barrier",
    ],
    "centerpoint": [
        "car", "truck", "construction_vehicle", "bus", "trailer",
        "barrier", "motorcycle", "bicycle", "pedestrian", "traffic_cone",
    ],
}[MODEL_NAME]

# Detection classes group the raw nuScenes categories shown on the right.
CATEGORY_FOR_DETECTION_CLASS = {
    "car": "vehicle.car",
    "truck": "vehicle.truck",
    "trailer": "vehicle.trailer",
    "bus": "vehicle.bus.rigid",
    "construction_vehicle": "vehicle.construction",
    "bicycle": "vehicle.bicycle",
    "motorcycle": "vehicle.motorcycle",
    "pedestrian": "human.pedestrian.adult",
    "traffic_cone": "movable_object.trafficcone",
    "barrier": "movable_object.barrier",
}
nuScenes_colors = get_colormap()
model_ontology = {
    class_name: {
        "idx": index,
        "rgb": list(nuScenes_colors[CATEGORY_FOR_DETECTION_CLASS[class_name]]),
    }
    for index, class_name in enumerate(MODEL_CLASS_NAMES)
}
dataset_to_model_translation = {
    class_name: class_name for class_name in dataset.ontology
}

MODEL_CONFIG_PATH.write_text(json.dumps(model_config, indent=2), encoding="utf-8")
MODEL_ONTOLOGY_PATH.write_text(json.dumps(model_ontology, indent=2), encoding="utf-8")
TRANSLATION_PATH.write_text(
    json.dumps(dataset_to_model_translation, indent=2), encoding="utf-8"
)
print(f"Saved model config to {MODEL_CONFIG_PATH}")
print(f"Saved model ontology to {MODEL_ONTOLOGY_PATH}")
print(f"Saved label translation to {TRANSLATION_PATH}")


## Load the MMDetection3D Model

MMDetection3D builds the selected model from its Python configuration and restores the downloaded checkpoint. `TorchLiDARDetectionModel` then provides the standard PerceptionMetrics inference and evaluation interface.

In [ ]:
mmdet3d_model = init_model(
    config=str(MMDET_CONFIG),
    checkpoint=str(CHECKPOINT),
    device=DEVICE,
)

model = TorchLiDARDetectionModel(
    model=mmdet3d_model,
    model_cfg=str(MODEL_CONFIG_PATH),
    ontology_fname=str(MODEL_ONTOLOGY_PATH),
    device=torch.device(DEVICE),
)

## Inspect One Sample

Point clouds use `x, y, z, intensity` features. Ground-truth and predicted boxes use `[x, y, z, dx, dy, dz, yaw]` in the LiDAR coordinate frame. The bird's-eye-view plot compares ground truth with predictions above the configured confidence threshold.

In [ ]:
def draw_bev_box(ax, box, color, text=None):
    x, y, _, dx, dy, _, yaw = box
    corners = np.array(
        [
            [-dx / 2, -dy / 2],
            [dx / 2, -dy / 2],
            [dx / 2, dy / 2],
            [-dx / 2, dy / 2],
            [-dx / 2, -dy / 2],
        ]
    )
    rotation = np.array(
        [[np.cos(yaw), -np.sin(yaw)], [np.sin(yaw), np.cos(yaw)]]
    )
    corners = corners @ rotation.T + np.array([x, y])
    ax.plot(corners[:, 0], corners[:, 1], color=color, linewidth=1.5)
    if text:
        ax.text(
            x,
            y,
            text,
            color=color,
            fontsize=8,
            ha="center",
            va="bottom",
            bbox={"facecolor": "white", "alpha": 0.7, "edgecolor": "none"},
        )

sample_id = dataset.dataset.index[0]
row = dataset.dataset.loc[sample_id]
points = dataset.read_points(row["points"])
gt_boxes, gt_labels = dataset.read_annotation(row["annotation"])
prediction = model.predict(points)

print(f"Sample: {sample_id}")
print(f"Points: {points.shape}")
print(f"Ground-truth boxes: {len(gt_boxes)}")
print(f"Predicted boxes: {len(prediction['boxes_3d'])}")

BEV_RANGE_METERS = 70

fig, ax = plt.subplots(figsize=(10, 10))
ax.scatter(points[:, 0], points[:, 1], c="lightgray", s=0.2)
class_names = {class_data["idx"]: class_name for class_name, class_data in dataset.ontology.items()}
for box, label in zip(gt_boxes, gt_labels):
    draw_bev_box(ax, box, "tab:green", class_names[int(label)])
for box, label, score in zip(
    prediction["boxes_3d"].numpy(),
    prediction["labels"].numpy(),
    prediction["scores"].numpy(),
):
    draw_bev_box(ax, box, "tab:red", f"{class_names[int(label)]} {score:.2f}")
ax.set_aspect("equal")
ax.set_xlim(-BEV_RANGE_METERS, BEV_RANGE_METERS)
ax.set_ylim(-BEV_RANGE_METERS, BEV_RANGE_METERS)
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("nuScenes LiDAR detection")
ax.legend(
    handles=[
        plt.Line2D([], [], color="tab:green", label="ground truth"),
        plt.Line2D([], [], color="tab:red", label="prediction"),
    ]
)
plt.show()

## Evaluate



In [ ]:
IOU_MODES = ("3d", "bev")

# Use a small subset for a quick tutorial run.
eval_dataset = copy(dataset)

from IPython.display import Markdown, display
import matplotlib.pyplot as plt

for iou_mode in IOU_MODES:
    results = model.eval(
        dataset=eval_dataset,
        split="val",
        iou_mode=iou_mode,
        ontology_translation=str(TRANSLATION_PATH),
        predictions_outdir=str(
            LOCAL_DIR / f"nuscenes_{MODEL_NAME}_predictions_{iou_mode}"
        ),
    )

    display(
        Markdown(
            f"### {iou_mode.upper()} IoU Detection Metrics "
            f"(threshold = {model.model_cfg.get('iou_threshold', 0.5):.2f})"
        )
    )
    display(results["metrics_df"])

